- California Housing (회귀)
- 입력(X): 캘리포니아 지역의 인구/주거 관련 특성(8개 수치형)
  예) MedInc(중위소득), HouseAge(주택연식), AveRooms(가구당 평균 방 수) 등
- 타깃(y): 주택 가격 중앙값(연속형)

목표
- 서로 다른 피처셋(기본/확장/전체)과 서로 다른 모델(선형회귀/랜덤포레스트)을
  동일한 전처리 흐름에서 평가하여 “가장 성능이 좋은 조합”을 자동 선택한다.

In [2]:
import numpy as np
import pandas as pd

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [3]:
# 1. 데이터 로드
data = fetch_california_housing(as_frame=True)

# 2. X/y 설정
X_all = data.data.copy()
y_all = data.target.copy()

In [4]:
# 3. 피처셋 정의
# base: 핵심 4개 피처
# plus: base + 2개 피처
# full: plus + 2개 피처(총 8개)
X_base_cols = ["MedInc","HouseAge","AveRooms","AveBedrms"]  # TODO: ["MedInc","HouseAge","AveRooms","AveBedrms"]
X_plus_cols = X_base_cols + ["Population","AveOccup"]  # TODO: base + ["Population","AveOccup"]
X_full_cols = X_plus_cols + ["Latitude","Longitude"]  # TODO: plus + ["Latitude","Longitude"]

feature_sets = [
    ("X_base", X_base_cols),
    ("X_plus", X_plus_cols),
    ("X_full", X_full_cols),
]

In [7]:
# 4. 공통 파이프라인
# 결측치 중앙값 대체 → 표준화 → 모델 학습을 하나의 Pipeline으로 구성합니다.
def make_pipe(model):
    return Pipeline(steps=[
        ("imp", SimpleImputer(strategy="median")),  # TODO: SimpleImputer(strategy="median")
        ("sc", StandardScaler()),   # TODO: StandardScaler()
        ("mdl", model),
    ])

In [8]:
# 5. 평가 함수
def evaluate(model_code, X, y, use_log=False, random_state=42):
    # (1) train/test split
    Xtr, Xte, ytr, yte = train_test_split(
        X,  # TODO
        y,  # TODO
        test_size=0.2,
        random_state=random_state
    )

    # (2) 모델 선택
    # LR: LinearRegression
    # RF: RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=...)
    if model_code == "LR":
        model = LinearRegression()  # TODO: LinearRegression()
    else:
        model = RandomForestRegressor()  # TODO: RandomForestRegressor(...)

    pipe = make_pipe(model)

    # (3) 학습 및 예측
    pipe.fit(Xtr, ytr)  # TODO: Xtr, ytr 로 학습
    yhat = pipe.predict(Xte)       # TODO: Xte 로 예측

    # (4) 평가 지표
    mae = mean_absolute_error(yte, yhat)  # TODO: mean_absolute_error(yte, yhat)
    mse = mean_squared_error(yte, yhat)  # TODO: mean_squared_error(yte, yhat)
    r2 = r2_score(yte, yhat)   # TODO: r2_score(yte, yhat)

    return {"MAE": mae, "MSE": mse, "R2": r2}

In [9]:
# 6. 실험 실행 (3 피처셋 × 2 모델 = 6조합)
experiments = []
models = [("LR", "LinearRegression"), ("RF", "RandomForestRegressor")]

for f_name, cols in feature_sets:
    X = X_all[cols].copy()
    y = y_all.copy()

    for m_code, m_name in models:
        scores = evaluate(m_code, X, y, random_state=42)
        scores.update({"Features": f_name, "Model": m_name, "Target": "y"})
        experiments.append(scores)

results = pd.DataFrame(experiments).sort_values(["MSE", "MAE"]).reset_index(drop=True)
print("=== 성능 요약 (낮을수록 좋음: MAE/MSE, 높을수록 좋음: R²) ===")
print(results)

=== 성능 요약 (낮을수록 좋음: MAE/MSE, 높을수록 좋음: R²) ===
        MAE       MSE        R2 Features                  Model Target
0  0.326277  0.252595  0.807240   X_full  RandomForestRegressor      y
1  0.462556  0.423490  0.676826   X_plus  RandomForestRegressor      y
2  0.528841  0.537515  0.589811   X_base  RandomForestRegressor      y
3  0.533200  0.555892  0.575788   X_full       LinearRegression      y
4  0.579214  0.642187  0.509934   X_plus       LinearRegression      y
5  0.580419  0.643568  0.508880   X_base       LinearRegression      y


In [10]:
# 7. 베스트 조합 자동 선택
best = results.iloc[0]
print("\n[Best]")
print(best)


[Best]
MAE                      0.326277
MSE                      0.252595
R2                        0.80724
Features                   X_full
Model       RandomForestRegressor
Target                          y
Name: 0, dtype: object
